# Discount Price Dataset

In [7]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

DATA_FILE = Path("all_catalogue_products.csv")
if not DATA_FILE.exists():
    DATA_FILE = Path("ML/Price-Prediction/price_prediction_by_shivam/all_catalogue_products.csv")

df = pd.read_csv(DATA_FILE, encoding="utf-8-sig")
print(f"Rows: {len(df):,}")
df.head()

Rows: 15,195


,retailer,region,catalogue_title,catalogue_start_date,catalogue_end_date,page_number,product_name,special_price,regular_price,save_amount,...,offer_text,ocr_confidence,needs_review,review_reason,source_page_image,source_catalogue_url,price_source,price_evidence,evidence_count,verification_status
0,Coles,VIC METRO,"Coles Catalogue January 1 - 7, 2025 VIC METRO",2025-01-01,2025-01-08,1,Peters Drumstick 4 Pack-6 Pack 475mL-490mL,4.75,9.5,4.75,...,WASS9.50 | WASS9.50 | Peters Drumstick | 4 Pac...,91.97,True,inferred_price_requires_review,https://caau.syd1.cdn.digitaloceanspaces.com/w...,https://www.catalogueau.com/coles/#catalogue=c...,page_discount_inference,page_discount,0,review
1,Coles,VIC METRO,"Coles Catalogue January 1 - 7, 2025 VIC METRO",2025-01-01,2025-01-08,1,Old El Paso Hard 'N Soft Taco Kit 350,4.75,9.5,4.75,...,WAS$9.50 | WAS$9.50 | Old El Paso Hard 'N Soft...,91.41,True,inferred_price_requires_review,https://caau.syd1.cdn.digitaloceanspaces.com/w...,https://www.catalogueau.com/coles/#catalogue=c...,page_discount_inference,page_discount,0,review
2,Coles,VIC METRO,"Coles Catalogue January 1 - 7, 2025 VIC METRO",2025-01-01,2025-01-08,1,Zooper Dooper Water Ice 24x70mL,3.60,7.2,3.60,...,WASS7.20 | WASS7.20 | Zooper Dooper Water Ice ...,91.78,True,inferred_price_requires_review,https://caau.syd1.cdn.digitaloceanspaces.com/w...,https://www.catalogueau.com/coles/#catalogue=c...,page_discount_inference,page_discount,0,review
3,Coles,VIC METRO,"Coles Catalogue January 1 - 7, 2025 VIC METRO",2025-01-01,2025-01-08,1,Gatorade Sports Drink or G Active Water 600mL,2.00,4.0,2.00,...,WASS4 | WASS4 | Gatorade Sports Drink or G Act...,94.70,False,NaN,https://caau.syd1.cdn.digitaloceanspaces.com/w...,https://www.catalogueau.com/coles/#catalogue=c...,displayed_price,displayed_price;page_discount;unit_price,2,verified
4,Coles,VIC METRO,"Coles Catalogue January 1 - 7, 2025 VIC METRO",2025-01-01,2025-01-08,2,Mars Chocolate Bar 30g-56g 2nd week,1.00,2.2,1.20,...,SAVE*1.20 | WASS2.20 | Mars Chocolate Bar 30g-...,95.23,False,NaN,https://caau.syd1.cdn.digitaloceanspaces.com/w...,https://www.catalogueau.com/coles/#catalogue=c...,save_was_arithmetic,displayed_price;save_was_arithmetic,2,verified


## Clean catalogue rows

In [8]:
data = df.copy()
data["catalogue_start_date"] = pd.to_datetime(data["catalogue_start_date"], errors="coerce")

number_columns = ["special_price", "regular_price", "save_amount", "discount_percent", "ocr_confidence", "evidence_count"]
for column in number_columns:
    data[column] = pd.to_numeric(data[column], errors="coerce")

def clean_name(name):
    name = str(name).lower().replace("&", " and ")
    name = re.sub(r"\bcoca[ -]?cola\b", "coca cola", name)
    name = re.sub(r"\bsoft drink\b", "", name)
    name = re.sub(r"\bkilograms?\b", "kg", name)
    name = re.sub(r"\bgrams?\b", "g", name)
    name = re.sub(r"\bmillilit(?:re|er)s?\b", "ml", name)
    name = re.sub(r"\blit(?:re|er)s?\b", "l", name)
    name = re.sub(r"[^a-z0-9.]+", " ", name)
    return re.sub(r"\s+", " ", name).strip()

data["original_product_name"] = data["product_name"].fillna("").str.strip()
data["canonical_product_name"] = data["original_product_name"].map(clean_name)
data = data[data["canonical_product_name"].str.count(r"[a-z]") >= 3].copy()
product_numbers = pd.Series(pd.factorize(data["canonical_product_name"])[0], index=data.index)
data["product_id"] = "P" + product_numbers.astype(str).str.zfill(5)
data["brand_key"] = data["canonical_product_name"].str.split().str[0]

calculated_regular = data["special_price"] + data["save_amount"]
data["regular_price_candidate"] = data["regular_price"].fillna(calculated_regular)
data["regular_price_origin"] = np.select(
    [data["regular_price"].notna(), data["regular_price"].isna() & calculated_regular.notna()],
    ["observed", "calculated"],
    default="missing",
)

trusted_source = data["price_source"].isin(["displayed_price", "save_was_arithmetic"])
trusted_review = data["verification_status"].eq("verified") | data["evidence_count"].ge(2)
special_ok = data["special_price"].between(0.01, 2_000)
special_trustworthy = trusted_source & trusted_review & data["ocr_confidence"].ge(85) & special_ok

calculated_discount = 100 * (data["regular_price_candidate"] - data["special_price"]) / data["regular_price_candidate"]
regular_ok = (
    data["regular_price_candidate"].between(0.01, 2_000)
    & data["regular_price_candidate"].ge(data["special_price"])
    & calculated_discount.between(0, 80)
)

data["special_price_clean"] = data["special_price"].where(special_trustworthy)
data["regular_price_clean"] = data["regular_price_candidate"].where(special_trustworthy & regular_ok)
data["price_trustworthy"] = data["special_price_clean"].notna().astype("int8")
data["price_outlier"] = (~special_ok | (data["regular_price_candidate"].notna() & ~regular_ok)).astype("int8")

key = ["retailer", "region", "product_id", "catalogue_start_date"]
before = len(data)
data = (
    data.sort_values(["price_trustworthy", "regular_price_clean", "evidence_count", "ocr_confidence"], ascending=False)
    .drop_duplicates(key)
    .sort_values(key)
    .reset_index(drop=True)
)

print(f"Valid products: {data['product_id'].nunique():,}")
print(f"Duplicates removed: {before - len(data):,}")
print(f"Trustworthy prices: {data['price_trustworthy'].sum():,}")
data[["product_id", "original_product_name", "special_price_clean", "regular_price_clean", "price_trustworthy"]].head()

Valid products: 6,784
Duplicates removed: 115
Trustworthy prices: 7,286


,product_id,original_product_name,special_price_clean,regular_price_clean,price_trustworthy
0,P00000,Peters Drumstick 4 Pack-6 Pack 475mL-490mL,NaN,NaN,0
1,P00000,Peters Drumstick 4 Pack-6 Pack 475mL-490mL,7.5,9.5,1
2,P00000,Peters Drumstick 4 Pack-6 Pack 475mL-490mL,NaN,NaN,0
3,P00000,Peters Drumstick 4 Pack-6 Pack 475mL-490mL,NaN,NaN,0
4,P00000,Peters Drumstick 4 Pack-6 Pack 475mL-490mL,7.5,9.5,1


## Build the final weekly dataset

In [9]:
weeks = pd.DataFrame({"catalogue_start_date": sorted(data["catalogue_start_date"].dropna().unique())})
weeks["week_index"] = range(len(weeks))

product_columns = ["retailer", "region", "product_id", "original_product_name", "canonical_product_name", "brand_key", "pack_size"]
products = data.sort_values("price_trustworthy", ascending=False).drop_duplicates(["retailer", "region", "product_id"])[product_columns]
weekly = products.merge(weeks, how="cross")

observed_columns = [
    "retailer", "region", "product_id", "catalogue_start_date", "special_price",
    "special_price_clean", "regular_price_clean", "regular_price_origin", "promo_type", "price_source",
    "price_trustworthy", "price_outlier",
]
weekly = weekly.merge(data[observed_columns], on=["retailer", "region", "product_id", "catalogue_start_date"], how="left", indicator=True)
weekly["catalogue_observed"] = weekly.pop("_merge").eq("both").astype("int8")
weekly["promo_type"] = weekly["promo_type"].fillna("none")
weekly["price_trustworthy"] = weekly["price_trustworthy"].fillna(0).astype("int8")
weekly["price_outlier"] = weekly["price_outlier"].fillna(0).astype("int8")

series_key = ["retailer", "region", "product_id"]
weekly = weekly.sort_values(series_key + ["catalogue_start_date"]).reset_index(drop=True)
group = weekly.groupby(series_key, sort=False)

last_regular_price = group["regular_price_clean"].ffill()
last_regular_week = weekly["week_index"].where(weekly["regular_price_clean"].notna()).groupby([weekly[column] for column in series_key]).ffill()
weekly["weeks_since_regular_price"] = weekly["week_index"] - last_regular_week
can_fill = weekly["weeks_since_regular_price"].between(1, 8)
weekly["regular_price_filled"] = weekly["regular_price_clean"].fillna(last_regular_price.where(can_fill))
weekly["regular_price_source"] = np.select(
    [
        weekly["regular_price_clean"].notna() & weekly["regular_price_origin"].eq("observed"),
        weekly["regular_price_clean"].notna() & weekly["regular_price_origin"].eq("calculated"),
        can_fill,
    ],
    ["observed", "calculated", "forward_filled"],
    default="missing",
)

weekly["is_special"] = (
    weekly["catalogue_observed"].eq(1)
    & weekly["promo_type"].isin(["half_price", "save_amount", "special"])
).astype("int8")
weekly["effective_price"] = weekly["special_price_clean"].fillna(weekly["regular_price_filled"])
weekly["price_inferred"] = (weekly["special_price_clean"].isna() & weekly["regular_price_filled"].notna()).astype("int8")
calculated_weekly_discount = 100 * (weekly["regular_price_filled"] - weekly["special_price_clean"]) / weekly["regular_price_filled"]
valid_weekly_discount = calculated_weekly_discount.between(0, 80) & weekly["is_special"].eq(1)
weekly["discount_percent"] = calculated_weekly_discount.where(valid_weekly_discount)
weekly.loc[weekly["is_special"].eq(0), "discount_percent"] = 0.0
weekly["discount_percent_trustworthy"] = (
    valid_weekly_discount
    & weekly["special_price_clean"].notna()
    & weekly["regular_price_clean"].notna()
).astype("int8")
weekly["history_count"] = group["price_trustworthy"].cumsum()
weekly["cold_start_product"] = weekly["history_count"].lt(3).astype("int8")

group = weekly.groupby(series_key, sort=False)
for lag in [1, 2, 4]:
    weekly[f"price_lag_{lag}"] = group["effective_price"].shift(lag)
weekly["discount_lag_1"] = group["discount_percent"].shift(1)
weekly["avg_price_4w"] = group["effective_price"].transform(lambda values: values.shift(1).rolling(4, min_periods=1).mean())
weekly["avg_price_8w"] = group["effective_price"].transform(lambda values: values.shift(1).rolling(8, min_periods=1).mean())
weekly["special_frequency_4w"] = group["is_special"].transform(lambda values: values.shift(1).rolling(4, min_periods=1).mean())
weekly["special_frequency_8w"] = group["is_special"].transform(lambda values: values.shift(1).rolling(8, min_periods=1).mean())

last_special_week = weekly["week_index"].where(weekly["is_special"].eq(1)).groupby([weekly[column] for column in series_key]).ffill()
weekly["weeks_since_last_special"] = weekly["week_index"] - last_special_week
weekly["week_of_year"] = weekly["catalogue_start_date"].dt.isocalendar().week.astype("int16")
weekly["month"] = weekly["catalogue_start_date"].dt.month.astype("int8")
weekly["quarter"] = weekly["catalogue_start_date"].dt.quarter.astype("int8")
weekly["season"] = weekly["month"].map({
    12: "summer", 1: "summer", 2: "summer",
    3: "autumn", 4: "autumn", 5: "autumn",
    6: "winter", 7: "winter", 8: "winter",
    9: "spring", 10: "spring", 11: "spring",
})

group = weekly.groupby(series_key, sort=False)
weekly["target_week"] = group["catalogue_start_date"].shift(-1)
weekly["target_is_special_next_week"] = group["is_special"].shift(-1)
weekly["target_discount_percent_next_week"] = group["discount_percent"].shift(-1)
target_discount_trustworthy = group["discount_percent_trustworthy"].shift(-1).eq(1)
weekly["target_discount_percent_next_week"] = weekly["target_discount_percent_next_week"].where(target_discount_trustworthy)
weekly["classification_eligible"] = weekly["target_is_special_next_week"].notna().astype("int8")
weekly["discount_regression_eligible"] = (
    weekly["target_is_special_next_week"].eq(1)
    & weekly["target_discount_percent_next_week"].notna()
).astype("int8")

keep = weekly["catalogue_observed"].eq(1) | weekly["effective_price"].notna()
final_columns = [
    "retailer", "region", "product_id", "original_product_name", "canonical_product_name", "brand_key", "pack_size",
    "catalogue_start_date", "week_index", "catalogue_observed", "promo_type", "special_price",
    "special_price_clean", "regular_price_filled", "regular_price_source", "weeks_since_regular_price",
    "effective_price", "price_inferred", "price_trustworthy", "price_outlier", "is_special", "discount_percent", "discount_percent_trustworthy",
    "history_count", "cold_start_product", "price_lag_1", "price_lag_2", "price_lag_4", "discount_lag_1",
    "avg_price_4w", "avg_price_8w", "special_frequency_4w", "special_frequency_8w", "weeks_since_last_special",
    "week_of_year", "month", "quarter", "season", "target_week",
    "target_is_special_next_week", "target_discount_percent_next_week",
    "classification_eligible", "discount_regression_eligible",
]
final = weekly.loc[keep, final_columns].copy()

assert final["product_id"].nunique() == data["product_id"].nunique()
assert not final.duplicated(["retailer", "region", "product_id", "catalogue_start_date"]).any()
assert final.loc[final["regular_price_source"].eq("forward_filled"), "weeks_since_regular_price"].between(1, 8).all()
assert final["season"].notna().all()
assert final.loc[final["classification_eligible"].eq(1), "target_is_special_next_week"].notna().all()
assert final.loc[final["discount_regression_eligible"].eq(1), "target_is_special_next_week"].eq(1).all()
assert final.loc[final["discount_regression_eligible"].eq(1), "target_discount_percent_next_week"].between(0, 80).all()

OUTPUT_FILE = DATA_FILE.parent / "discount_price_final_dataset.csv"
final.to_csv(OUTPUT_FILE, index=False)

print(f"Final rows: {len(final):,}")
print(f"Products retained: {final['product_id'].nunique():,}")
print(f"Classification rows: {final['classification_eligible'].sum():,}")
print(f"Next-week specials: {int(final.loc[final['classification_eligible'].eq(1), 'target_is_special_next_week'].sum()):,}")
print(f"Discount regression rows: {final['discount_regression_eligible'].sum():,}")
print(f"Saved: {OUTPUT_FILE}")
final.head()

Final rows: 45,431
Products retained: 6,784
Classification rows: 44,640
Next-week specials: 4,192
Discount regression rows: 2,376
Saved: discount_price_final_dataset.csv


,retailer,region,product_id,original_product_name,canonical_product_name,brand_key,pack_size,catalogue_start_date,week_index,catalogue_observed,...,weeks_since_last_special,week_of_year,month,quarter,season,target_week,target_is_special_next_week,target_discount_percent_next_week,classification_eligible,discount_regression_eligible
0,Coles,VIC METRO,P00000,Peters Drumstick 4 Pack-6 Pack 475mL-490mL,peters drumstick 4 pack 6 pack 475ml 490ml,peters,4 Pack; 6 Pack; 475mL; 490mL,2025-01-01,0,1,...,0.0,1,1,1,summer,2025-01-08,0.0,NaN,1,0
2,Coles,VIC METRO,P00000,Peters Drumstick 4 Pack-6 Pack 475mL-490mL,peters drumstick 4 pack 6 pack 475ml 490ml,peters,4 Pack; 6 Pack; 475mL; 490mL,2025-01-15,2,1,...,0.0,3,1,1,summer,2025-01-22,0.0,NaN,1,0
3,Coles,VIC METRO,P00000,Peters Drumstick 4 Pack-6 Pack 475mL-490mL,peters drumstick 4 pack 6 pack 475ml 490ml,peters,4 Pack; 6 Pack; 475mL; 490mL,2025-01-22,3,0,...,1.0,4,1,1,summer,2025-01-29,1.0,NaN,1,0
4,Coles,VIC METRO,P00000,Peters Drumstick 4 Pack-6 Pack 475mL-490mL,peters drumstick 4 pack 6 pack 475ml 490ml,peters,4 Pack; 6 Pack; 475mL; 490mL,2025-01-29,4,1,...,0.0,5,1,1,summer,2025-02-05,0.0,NaN,1,0
5,Coles,VIC METRO,P00000,Peters Drumstick 4 Pack-6 Pack 475mL-490mL,peters drumstick 4 pack 6 pack 475ml 490ml,peters,4 Pack; 6 Pack; 475mL; 490mL,2025-02-05,5,0,...,1.0,6,2,1,summer,2025-02-12,0.0,NaN,1,0
